<a href="https://colab.research.google.com/github/olumideadekunle/Building-For-AI/blob/main/Agent_Building_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build and Deploy AI Agents with Gemini and BigQuery MCP Server

This notebook guides you through setting up an AI agent using the Agent Development Kit (ADK) that interacts with BigQuery via the Model Context Protocol (MCP) server.

## 1. Install Agent Development Kit (ADK) and `uv`

First, we'll install the necessary libraries, including `google-adk` and `uv` (a fast Python package installer and executor, often used with ADK).

In [18]:
# Uninstall potentially incompatible versions first
!uv pip uninstall -y google-adk mcp mcp-types

# Install google-adk with its mcp extra, which should handle mcp and mcp-types dependencies.
# Also install uv and nest_asyncio separately as they are not dependencies of google-adk[mcp].
!uv pip install google-adk[mcp] uv nest_asyncio

Using Python 3.12.13 environment at: /usr
Uninstalled 1 package in 149ms
 - google-adk==2.4.0
Using Python 3.12.13 environment at: /usr
Resolved 65 packages in 526ms
Prepared 5 packages in 235ms
Uninstalled 1 package in 27ms
Installed 5 packages in 26ms
 + google-adk==2.7.0
 - google-genai==2.11.0
 + google-genai==2.12.1
 + httpx-sse==0.4.3
 + mcp==1.29.0
 + pydantic-settings==2.15.0


## 2. Set up Google API Key for Gemini

The `LlmAgent` uses the Gemini model, which requires a Google API Key. It's best practice to store this securely in Colab's **Secrets manager**.

**Action Required:**
1.  Click the '🔑' icon (Secrets) in the left sidebar of your Colab notebook.
2.  Click '+ New secret'.
3.  For the **Name**, type `GOOGLE_API_KEY`.
4.  For the **Value**, paste your actual Gemini API key (you can get one from [Google AI Studio](https://aistudio.google.com/app/apikey)).
5.  Crucially, ensure the **'Notebook access' toggle is switched ON** for this secret.

Once set, this cell will retrieve and use your API key as an environment variable.

In [11]:
import os
from google.colab import userdata

# Retrieve GOOGLE_API_KEY from Colab Secrets
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    %env GOOGLE_API_KEY=$GOOGLE_API_KEY
    print("GOOGLE_API_KEY successfully loaded from Colab Secrets.")
except Exception as e:
    print(f"Error retrieving GOOGLE_API_KEY: {e}")
    print("Please ensure you have added 'GOOGLE_API_KEY' to Colab Secrets and enabled 'Notebook access'.")
    GOOGLE_API_KEY = None # Ensure it's None if not set

Error retrieving GOOGLE_API_KEY: Secret GOOGLE_API_KEY does not exist.
Please ensure you have added 'GOOGLE_API_KEY' to Colab Secrets and enabled 'Notebook access'.


## 3. Configure Google Cloud Project and Authenticate

The Data Agent will access BigQuery using your Google Cloud credentials. You need to specify your Google Cloud Project ID and a Cloud Run region. We'll use `google.colab.auth.authenticate_user()` for robust authentication within Colab.

**Action Required:**
*   **Replace `'YOUR_PROJECT_ID'`** with your actual Google Cloud Project ID.
*   **Replace `'CLOUD-RUN-REGION'`** with a valid Cloud Run region (e.g., `'us-central1'`).

When running this cell, a pop-up window might appear for Google account authentication. Please follow the prompts to complete it.

In [12]:
from google.colab import auth
import os

# Authenticate to Google Cloud
print("Authenticating Google Cloud user...")
auth.authenticate_user()
print("Google Cloud authentication successful.")

# >>> REPLACE 'YOUR_PROJECT_ID' AND 'CLOUD-RUN-REGION' WITH YOUR ACTUAL VALUES <<<
YOUR_PROJECT_ID = 'YOUR_PROJECT_ID'  # <--- REPLACE THIS WITH YOUR GOOGLE CLOUD PROJECT ID
CLOUD_RUN_REGION = 'us-central1' # <--- REPLACE THIS (e.g., 'us-central1')

# Set gcloud configuration and environment variables
print(f"Setting gcloud project to: {YOUR_PROJECT_ID}")
!gcloud config set project {YOUR_PROJECT_ID}
print(f"Setting gcloud run region to: {CLOUD_RUN_REGION}")
!gcloud config set run/region {CLOUD_RUN_REGION}

%env GOOGLE_CLOUD_PROJECT={YOUR_PROJECT_ID}
%env GOOGLE_CLOUD_REGION={CLOUD_RUN_REGION}
%env GOOGLE_GENAI_USE_ENTERPRISE=True # Use Agent Platform
%env GOOGLE_CLOUD_LOCATION=global # Use global Gemini API endpoint

print(f"\nConfigured Google Cloud Project: {os.environ.get('GOOGLE_CLOUD_PROJECT')}")
print(f"Configured Google Cloud Region: {os.environ.get('GOOGLE_CLOUD_REGION')}")

Authenticating Google Cloud user...
Google Cloud authentication successful.
Setting gcloud project to: YOUR_PROJECT_ID
Are you sure you wish to set property [core/project] to YOUR_PROJECT_ID?

Do you want to continue (Y/n)?  Y

ERROR: (gcloud.config.set) The project property must be set to a valid project ID, not the project name [YOUR_PROJECT_ID]
To set your project, run:

  $ gcloud config set project PROJECT_ID

or to unset it, run:

  $ gcloud config unset project
Setting gcloud run region to: us-central1
Updated property [run/region].
env: GOOGLE_CLOUD_PROJECT=YOUR_PROJECT_ID
env: GOOGLE_CLOUD_REGION=us-central1
env: GOOGLE_GENAI_USE_ENTERPRISE=True # Use Agent Platform
env: GOOGLE_CLOUD_LOCATION=global # Use global Gemini API endpoint

Configured Google Cloud Project: YOUR_PROJECT_ID
Configured Google Cloud Region: us-central1


## 4. Enable Required Google Cloud APIs

We need to enable several Google Cloud APIs for the agent to function correctly. This step can take a few minutes.

In [13]:
PROJECT_ID = os.environ.get('GOOGLE_CLOUD_PROJECT')

if not PROJECT_ID or PROJECT_ID == 'YOUR_PROJECT_ID':
    print("ERROR: GOOGLE_CLOUD_PROJECT is not set or is still the placeholder. Please set it in the previous cell.")
else:
    print(f"Enabling services for project: {PROJECT_ID}")
    !gcloud services enable --project "{PROJECT_ID}" \
        run.googleapis.com \
        cloudbuild.googleapis.com \
        artifactregistry.googleapis.com \
        bigquery.googleapis.com \
        aiplatform.googleapis.com
    print("API enablement request sent. This may take a few minutes to take effect.")

ERROR: GOOGLE_CLOUD_PROJECT is not set or is still the placeholder. Please set it in the previous cell.


## 5. Create Data Agent Files

Now, we'll create the `data_agent` directory and its essential files: `agent.py`, `__init__.py`, and `requirements.txt`.

In [14]:
# Clean up any previous 'data_agent' directory
!rm -rf data_agent

# Create the root directory for the agent
!mkdir data_agent

# Create __init__.py
!echo "from . import agent" > data_agent/__init__.py

# Create requirements.txt with specified versions
!echo -e "google-adk==2.4.*\nmcp==1.29.*" > data_agent/requirements.txt

print("Data agent directory and basic files created:")
!ls -R data_agent

Data agent directory and basic files created:
data_agent:
__init__.py  requirements.txt


In [15]:
data_agent_code = '''import os
import nest_asyncio
nest_asyncio.apply() # Apply to allow asyncio to run in Colab

from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams

import google.auth
from google.auth.transport.requests import Request

# Fetch Application Default Credentials (ADC)
# to use as agent's own identity for accessing BigQuery MCP Server
_application_default_credentials, project_id_adc = google.auth.default()
_request = Request()
_application_default_credentials.refresh(_request)

# Retrieve Google Cloud project to use from environment variable
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
if not project_id:
    raise ValueError("GOOGLE_CLOUD_PROJECT environment variable is not set. Please set it in the Colab cell for GCP configuration.")

# Builds authentication headers for MCP Server requests,
# and refreshes credentials if needed.
def _adc_auth_header_provider(context = None) -> dict[str, str]:
    if not _application_default_credentials.valid:
        _application_default_credentials.refresh(_request)

    return {
        "Authorization": f"Bearer {_application_default_credentials.token}",
        "x-goog-user-project": project_id
    }

# Initialize the MCP Toolset with the connection parameters
bigquery_toolset = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url="https://bigquery.googleapis.com/mcp",
        tool_filter=[
            'get_dataset_info',
            'list_table_ids',
            'get_table_info',
            # Using readonly is a security measure to prevent accidental data modification.
            'execute_sql_readonly',
        ]
    ),
    header_provider=_adc_auth_header_provider # Auth header provider function
)

# Configure the agent
system_instruction = f"""
You are a helpful assistant that can answer questions about data in BigQuery.
To answer the user's question, use data you have access to by using tools `list_table_ids` and `get_table_info`.
Your data is in `bigquery-public-data.new_york_citibike` dataset (Citi Bike trips and stations in the NYC area.)

Plan of action:
0. ALWAYS start by analyzing dataset.
1. Analyze your data, investigate schema and dimensions by querying distrinct values of columns using `execute_sql_readonly`.
   Output information about tables, columns, their data types and sets of values (for dimensions).
   Note which columns can be joined or used in aggregations/filters, and what type conversion may be needed for joining or aggregating.
   DO NOT MAKE ASSUMPTIONS ABOUT DATA (structure, type, values, relationships) BASED ON YOUR PRIOR KNOWLEDGE. ALWAYS VERIFY YOUR ASSUMPTIONS.
2. Understand and interpret the user's question.
3. Formulate a plan to answer the user's question.
4. Write a SQL query to retrieve relevant data in necessary form.
   This is where you must pay extra attention to column types and dimensions' sets of values.
5. Retrieve data by generating BigQuery SQL and using `execute_sql_readonly`.
   Always use Dry Run to verify SQL correctness.
   Use `{project_id}` to run BigQuery queries (`project_id` parameter of `execute_sql_readonly`).

Do not use LaTeX in your responses. When giving a final answer, use Markdown.
"""

root_agent = LlmAgent(
    model="gemini-3.6-flash", # This model requires GOOGLE_API_KEY to be set
    name="data_agent",
    instruction=system_instruction,
    description="A helpful assistant that can answer questions using NYC Citibike data.",
    tools=[bigquery_toolset]
)'''

# Write the content to the agent.py file
with open('data_agent/agent.py', 'w') as f:
    f.write(data_agent_code)

print('data_agent/agent.py has been created.')
# !cat data_agent/agent.py # Uncomment to display content for verification

data_agent/agent.py has been created.


## 6. Run the Data Agent Locally

Now you can run your `data_agent` in an interactive command-line interface. For local testing in Colab, `adk run` is generally more straightforward than `adk web`, which requires additional setup like `ngrok` for external access.

**To interact:** Once the agent starts, you'll see a prompt `[user]:`. Type your questions (e.g., `What data do you have?` or `What are the top 5 stations by number of trips?`) and press Enter.

**To exit:** Type `exit` or press `Ctrl+C` in the prompt, or use the square stop button in Colab to stop the cell execution.

In [16]:
# Run the agent in command-line interface
print("Starting ADK agent in command-line mode. Type 'exit' to quit.")

# This command will block execution until you type 'exit' or stop the cell.
!adk run data_agent

Starting ADK agent in command-line mode. Type 'exit' to quit.
Log setup complete: /tmp/agents_log/agent.20260814_110957.log
To access latest log: tail -F /tmp/agents_log/agent.latest.log
Traceback (most recent call last):
  File "/usr/local/bin/adk", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1569, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1490, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1970, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1353, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

### Optional: Run with Web Interface (Advanced Colab Setup)

If you prefer the ADK Web UI, you would typically use `adk web`. However, running this in Colab usually requires setting up port forwarding (e.g., using `ngrok`) to make the local web server accessible in your browser. This is beyond the scope of this basic setup, but the command would look something like this:

```python
# This requires ngrok setup for external access to the Colab runtime
# !uv tool run --with "mcp==1.29.*" --from "google-adk[mcp]==2.4.*" adk web --allow_origins="*" --port 8081 .
```

## ADK Python Quickstart in Colab

This guide will walk you through setting up and running your first agent using the Agent Development Kit (ADK) for Python, adapted for Google Colab.

### 1. Installation

First, we need to install the `google-adk` library. We'll also install `nest_asyncio` as it's often helpful for running asyncio code in environments like Colab.

In [2]:
!pip install google-adk nest_asyncio

### 2. Create an Agent Project

We'll create a new agent project named `my_agent` using the `adk create` command. This will create a directory with a basic `agent.py` file.

In [3]:
# Remove any existing 'my_agent' directory to ensure a clean start
!rm -rf my_agent

# Manually create the agent project directory as adk create is interactive
!mkdir my_agent

# Confirm directory creation
!ls -d my_agent

my_agent


### 3. Update your Agent Project

The guide suggests updating the `agent.py` file to include a `get_current_time` tool. We will write the updated content directly to `my_agent/agent.py`.

In [4]:
agent_code = '''from google.adk.agents.llm_agent import Agent
import nest_asyncio
nest_asyncio.apply()

# Mock tool implementation
def get_current_time(city: str) -> dict:
    """Returns the current time in a specified city."""
    # In a real scenario, this would fetch actual time
    import datetime
    import pytz

    try:
        tz = pytz.timezone(city.replace(' ', '_')) # Basic attempt to get timezone
        current_time = datetime.datetime.now(tz).strftime('%H:%M %p %Z')
        return {"status": "success", "city": city, "time": current_time}
    except pytz.exceptions.UnknownTimeZoneError:
        return {"status": "error", "message": f"Unknown timezone for city: {city}"}
    except Exception as e:
        return {"status": "error", "message": str(e)}

root_agent = Agent(
    model='gemini-flash-latest',
    name='root_agent',
    description="Tells the current time in a specified city.",
    instruction="You are a helpful assistant that tells the current time in cities. Use the 'get_current_time' tool for this purpose.",
    tools=[get_current_time],
)'''

# Write the content to the agent.py file
with open('my_agent/agent.py', 'w') as f:
    f.write(agent_code)

print('my_agent/agent.py has been updated.')
!cat my_agent/agent.py # Display the content for verification

my_agent/agent.py has been updated.
from google.adk.agents.llm_agent import Agent
import nest_asyncio
nest_asyncio.apply()

# Mock tool implementation
def get_current_time(city: str) -> dict:
    """Returns the current time in a specified city."""
    # In a real scenario, this would fetch actual time
    import datetime
    import pytz

    try:
        tz = pytz.timezone(city.replace(' ', '_')) # Basic attempt to get timezone
        current_time = datetime.datetime.now(tz).strftime('%H:%M %p %Z')
        return {"status": "success", "city": city, "time": current_time}
    except pytz.exceptions.UnknownTimeZoneError:
        return {"status": "error", "message": f"Unknown timezone for city: {city}"}
    except Exception as e:
        return {"status": "error", "message": str(e)}

root_agent = Agent(
    model='gemini-flash-latest',
    name='root_agent',
    description="Tells the current time in a specified city.",
    instruction="You are a helpful assistant that tells the current 

In [9]:
# Import necessary libraries
from google.colab import userdata
import os

# Get the API key from Colab secrets
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception as e:
    print(f"Error retrieving GOOGLE_API_KEY: {e}")
    print("Please ensure you have added 'GOOGLE_API_KEY' to Colab Secrets (🔑 icon on the left sidebar) and enabled 'Notebook access'.")
    GOOGLE_API_KEY = None

# Create the .env file within the my_agent directory only if API key is available and directory exists
if GOOGLE_API_KEY and os.path.exists('my_agent'):
    env_content = f'GOOGLE_API_KEY="{GOOGLE_API_KEY}"'
    with open('my_agent/.env', 'w') as f:
        f.write(env_content)
    print('my_agent/.env has been created with your GOOGLE_API_KEY.')
else:
    if not GOOGLE_API_KEY:
        print("Skipping .env creation because GOOGLE_API_KEY is not set.")
    if not os.path.exists('my_agent'):
        print("Skipping .env creation because 'my_agent' directory does not exist.")

# You can uncomment the line below to verify the content (be cautious with sharing output containing your key)
# !cat my_agent/.env

Error retrieving GOOGLE_API_KEY: Secret GOOGLE_API_KEY does not exist.
Please ensure you have added 'GOOGLE_API_KEY' to Colab Secrets (🔑 icon on the left sidebar) and enabled 'Notebook access'.
Skipping .env creation because GOOGLE_API_KEY is not set.


### 4. Set your API Key

ADK with Gemini API requires an API key. In Colab, it's best practice to store your API key in the **Secrets** manager (click the '🔑' icon in the left sidebar).

1. Add a new secret.
2. Name it `GOOGLE_API_KEY`.
3. Paste your Gemini API key as the value.
4. Enable 'Notebook access' for this secret.

After adding the secret, we will create an `.env` file in the `my_agent` directory to make it accessible to the ADK runtime.

### 5. Run your Agent

Now you can run your ADK agent using the command-line interface. We need to execute the command from the parent directory of `my_agent`.

In [6]:
# Navigate to the parent directory and run the agent
import os

# Ensure we are in the directory containing 'my_agent'
# The current working directory is typically /content in Colab
# If you've changed directories, adjust this path accordingly.
if not os.path.exists('my_agent/agent.py'):
    print("Error: 'my_agent/agent.py' not found. Please ensure the 'my_agent' directory is in the current working directory.")
else:
    print("Running the agent. Type your questions in the input field below.")
    print("Example: What time is it in London?")
    print("To exit, type 'exit' or press Ctrl+C.")
    # The !adk run command will start an interactive session.
    # Colab might require stopping the cell execution manually (square stop button).
    !adk run my_agent

Running the agent. Type your questions in the input field below.
Example: What time is it in London?
To exit, type 'exit' or press Ctrl+C.
Log setup complete: /tmp/agents_log/agent.20260814_110342.log
To access latest log: tail -F /tmp/agents_log/agent.latest.log
/usr/local/lib/python3.12/dist-packages/google/adk/cli/cli.py:334: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
Running agent root_agent, type exit to exit.
[user]: 
Aborted!
^C


---

## Create a Data Agent using Agent Development Kit

Now, let's create a more advanced Data Agent that can interact with BigQuery using the Model Context Protocol (MCP) server. This agent will use your Google Cloud credentials to access BigQuery data.

### 1. Setup and Requirements (Google Cloud Project)

For this Data Agent, you need to set your Google Cloud Project ID and a Cloud Run region. The agent will use your Google Cloud credentials to access BigQuery. It's recommended to run `gcloud auth login` in a separate terminal or ensure your Colab environment is authenticated to GCP.

**Action Required:**
*   Replace `YOUR_PROJECT_ID` with your actual Google Cloud Project ID.
*   Optionally, replace `CLOUD-RUN-REGION` with a supported Cloud Run region (e.g., `us-central1`).

In [7]:
# Authenticate to Google Cloud for ADC. Colab provides an easy way to do this.
from google.colab import auth
auth.authenticate_user()

# Set your Google Cloud Project ID and Region
# >>> REPLACE 'YOUR_PROJECT_ID' AND 'CLOUD-RUN-REGION' WITH YOUR ACTUAL VALUES <<<
YOUR_PROJECT_ID = 'YOUR_PROJECT_ID'  # <--- REPLACE THIS WITH YOUR GOOGLE CLOUD PROJECT ID
CLOUD_RUN_REGION = 'CLOUD-RUN-REGION' # <--- REPLACE THIS (e.g., 'us-central1')

# Set project ID for gcloud config and environment variables
!gcloud config set project {YOUR_PROJECT_ID}
!gcloud config set run/region {CLOUD_RUN_REGION}

# Set environment variables for the agent script
%env GOOGLE_CLOUD_PROJECT={YOUR_PROJECT_ID}
%env GOOGLE_CLOUD_REGION={CLOUD_RUN_REGION}
%env GOOGLE_GENAI_USE_ENTERPRISE=True # Use Agent Platform
%env GOOGLE_CLOUD_LOCATION=global # Use global Gemini API endpoint

print(f"Google Cloud Project: {os.environ.get('GOOGLE_CLOUD_PROJECT')}")
print(f"Google Cloud Region: {os.environ.get('GOOGLE_CLOUD_REGION')}")

Are you sure you wish to set property [core/project] to YOUR_PROJECT_ID?

Do you want to continue (Y/n)?  

Command killed by keyboard interrupt

^C
Updated property [run/region].
env: GOOGLE_CLOUD_PROJECT=YOUR_PROJECT_ID
env: GOOGLE_CLOUD_REGION=CLOUD-RUN-REGION
env: GOOGLE_GENAI_USE_ENTERPRISE=True # Use Agent Platform
env: GOOGLE_CLOUD_LOCATION=global # Use global Gemini API endpoint
Google Cloud Project: YOUR_PROJECT_ID
Google Cloud Region: CLOUD-RUN-REGION


### 2. Enable APIs

Next, we need to enable the Google Cloud APIs required for this codelab. This step might take a few minutes to take effect.

In [19]:
PROJECT_ID = os.environ.get('GOOGLE_CLOUD_PROJECT')

if not PROJECT_ID or PROJECT_ID == 'YOUR_PROJECT_ID':
    print("ERROR: GOOGLE_CLOUD_PROJECT is not set or is still the placeholder. Please set it in the previous cell.")
else:
    print(f"Enabling services for project: {PROJECT_ID}")
    !gcloud services enable --project "{PROJECT_ID}" \
        run.googleapis.com \
        cloudbuild.googleapis.com \
        artifactregistry.googleapis.com \
        bigquery.googleapis.com \
        aiplatform.googleapis.com

ERROR: GOOGLE_CLOUD_PROJECT is not set or is still the placeholder. Please set it in the previous cell.


### 3. Write Agent's Code

Now, we'll create the `data_agent` directory and its required files (`agent.py`, `__init__.py`, `requirements.txt`).

In [20]:
# Remove any existing 'data_agent' directory to ensure a clean start
!rm -rf data_agent

# Create the root directory for the agentic app
!mkdir data_agent

# Create __init__.py
!echo "from . import agent" > data_agent/__init__.py

# Create requirements.txt
!echo -e "google-adk==2.4.*\nmcp==1.29.*" > data_agent/requirements.txt

# Display the created structure
!ls -R data_agent

data_agent:
__init__.py  requirements.txt


In [21]:
data_agent_code = '''import os

from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams

import google.auth
from google.auth.transport.requests import Request

# Fetch Application Default Credentials (ADC)
# to use as agent's own identity for accessing BigQuery MCP Server
_application_default_credentials, project_id = google.auth.default()
_request = Request()
_application_default_credentials.refresh(_request)

# Retrieve Google Cloud project to use.
project_id = os.getenv("GOOGLE_CLOUD_PROJECT", project_id)
if not project_id:
    raise ValueError("GOOGLE_CLOUD_PROJECT environment variable is not set.")

# Builds authentication headers for MCP Server requests,
# and refreshes credentials if needed.
def _adc_auth_header_provider(context = None) -> dict[str, str]:
    if not _application_default_credentials.valid:
        _application_default_credentials.refresh(_request)

    return {
        "Authorization": f"Bearer {_application_default_credentials.token}",
        "x-goog-user-project": project_id
    }

# Initialize the MCP Toolset with the connection parameters
bigquery_toolset = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url="https://bigquery.googleapis.com/mcp",
        tool_filter=[
            'get_dataset_info',
            'list_table_ids',
            'get_table_info',
            # Using readonly is a security measure to prevent accidental data modification.
            'execute_sql_readonly',
        ]
    ),
    header_provider=_adc_auth_header_provider # Auth header provider function
)

# Configure the agent

system_instruction = f"""
You are a helpful assistant that can answer questions about data in BigQuery.
To answer the user's question, use data you have access to by using tools `list_table_ids` and `get_table_info`.
Your data is in `bigquery-public-data.new_york_citibike` dataset (Citi Bike trips and stations in the NYC area.)

Plan of action:
0. ALWAYS start by analyzing dataset.
1. Analyze your data, investigate schema and dimensions by querying distrinct values of columns using `execute_sql_readonly`.
   Output information about tables, columns, their data types and sets of values (for dimensions).
   Note which columns can be joined or used in aggregations/filters, and what type conversion may be needed for joining or aggregating.
   DO NOT MAKE ASSUMPTIONS ABOUT DATA (structure, type, values, relationships) BASED ON YOUR PRIOR KNOWLEDGE. ALWAYS VERIFY YOUR ASSUMPTIONS.
2. Understand and interpret the user's question.
3. Formulate a plan to answer the user's question.
4. Write a SQL query to retrieve relevant data in necessary form.
   This is where you must pay extra attention to column types and dimensions' sets of values.
5. Retrieve data by generating BigQuery SQL and using `execute_sql_readonly`.
   Always use Dry Run to verify SQL correctness.
   Use `{project_id}` to run BigQuery queries (`project_id` parameter of `execute_sql_readonly`).

Do not use LaTeX in your responses. When giving a final answer, use Markdown.
"""

root_agent = LlmAgent(
    model="gemini-3.6-flash",
    name="data_agent",
    instruction=system_instruction,
    description="A helpful assistant that can answer questions using NYC Citibike data.",
    tools=[bigquery_toolset]
)'''

# Write the content to the agent.py file
with open('data_agent/agent.py', 'w') as f:
    f.write(data_agent_code)

print('data_agent/agent.py has been created.')
!cat data_agent/agent.py # Display the content for verification

data_agent/agent.py has been created.
import os

from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams

import google.auth
from google.auth.transport.requests import Request

# Fetch Application Default Credentials (ADC)
# to use as agent's own identity for accessing BigQuery MCP Server
_application_default_credentials, project_id = google.auth.default()
_request = Request()
_application_default_credentials.refresh(_request)

# Retrieve Google Cloud project to use.
project_id = os.getenv("GOOGLE_CLOUD_PROJECT", project_id)
if not project_id:
    raise ValueError("GOOGLE_CLOUD_PROJECT environment variable is not set.")

# Builds authentication headers for MCP Server requests,
# and refreshes credentials if needed.
def _adc_auth_header_provider(context = None) -> dict[str, str]:
    if not _application_default_credentials.valid:
        _application_defau

### 4. Try the agent locally

Now you can try to run the `data_agent`. The codelab suggests using `adk web` for an interactive web interface. However, running `adk web` in Colab directly will expose a local port that is not directly accessible without further setup like `ngrok` for port forwarding.

For simplicity in Colab, it's often easier to use `adk run` for a command-line interactive session. If you wish to use the `adk web` interface, you will need to set up port forwarding. I will provide the `adk web` command as per your codelab, but note the Colab specific steps.

**Note:** Running this command will start a server. You will need to stop the cell execution manually (using the square stop button in Colab) to proceed after testing.

In [ ]:
# To run the web interface (requires port forwarding like ngrok for full access in Colab):
# First, install uv if you don't have it (this might be needed for the specific command format):
!pip install uv

# The command to start the web UI. Trying port 8081 to avoid conflicts.
print("Starting ADK Web interface on port 8081. If running in Colab, you might need ngrok for external access.")
print("To interact, you can also use 'adk run data_agent' for a command-line interface in a new cell after this one.")

# This command will block execution. You will need to stop the cell manually.
!uv tool run --with "mcp==1.29.*" --from "google-adk[mcp]==2.4.*" adk web --allow_origins="*" --port 8081 .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 42.0 MB/s eta 0:00:00
Starting ADK Web interface on port 8081. If running in Colab, you might need ngrok for external access.
To interact, you can also use 'adk run data_agent' for a command-line interface in a new cell after this one.
Installed 54 packages in 90ms
2026-08-14 11:32:09,036 - INFO - service_factory.py:266 - Using in-memory memory service
2026-08-14 11:32:09,036 - INFO - local_storage.py:88 - Using per-agent session storage rooted at /content
2026-08-14 11:32:09,036 - INFO - local_storage.py:120 - Using per-agent artifact storage rooted at /content
/root/.cache/uv/archive-v0/JWqCKgsYpJpgPn99/lib/python3.12/site-packages/google/adk/cli/fast_api.py:563: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service = InMemoryCredentialService()
/root/.cache/uv/archive

### Next Steps

You have successfully set up and run a basic ADK agent! You can now experiment with modifying the `agent.py` file to add more tools or change the agent's instructions. Remember, ADK also supports a web interface, but setting that up in Colab typically requires additional steps like ngrok for port forwarding.